In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tabpfn import TabPFNClassifier
from tabpfn_extensions.embedding import TabPFNEmbedding

In [ ]:

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
class StudentModel(nn.Module):
    def __init__(self, in_dim, emb_dim, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            #nn.Linear(256, 256),
            #nn.ReLU(),
            nn.Linear(256, emb_dim),
        )
        self.decoder = nn.Linear(emb_dim, n_classes)

    def forward(self, x):
        z = self.encoder(x)
        logits = self.decoder(z)
        return z, logits

In [ ]:
teacher = TabPFNClassifier(n_estimators=1, random_state=42)
teacher.fit(X_train, y_train)

embedder = TabPFNEmbedding(model=teacher, n_fold=0)

emb_train = embedder.fit_transform(X_train, y_train)
emb_test = embedder.transform(X_test)

if emb_train.ndim == 3:
    emb_train = emb_train[0]
    emb_test = emb_test[0]

teacher_proba_train = teacher.predict_proba(X_train)
teacher_proba_test = teacher.predict_proba(X_test)

x_scaler = StandardScaler().fit(X_train)
emb_scaler = StandardScaler().fit(emb_train)

X_train_s = x_scaler.transform(X_train)
X_test_s = x_scaler.transform(X_test)

emb_train_s = emb_scaler.transform(emb_train)
emb_test_s = emb_scaler.transform(emb_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

emb_train_t = torch.tensor(emb_train_s, dtype=torch.float32)
emb_test_t = torch.tensor(emb_test_s, dtype=torch.float32)

teacher_train_t = torch.tensor(teacher_proba_train, dtype=torch.float32)

n_features = X_train.shape[1]
emb_dim = emb_train.shape[1]
n_classes = teacher_proba_train.shape[1]


student = StudentModel(n_features, emb_dim, n_classes)

opt1 = torch.optim.AdamW(student.encoder.parameters(), lr=1e-3, weight_decay=1e-4)
opt2 = torch.optim.AdamW(student.decoder.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
alpha = 1.0
beta = 1.0

for epoch in range(2000):
    opt1.zero_grad()
    opt2.zero_grad()

    z_hat, logits_hat = student(X_train_t)
    log_probs_hat = F.log_softmax(logits_hat, dim=1)

    loss_emb = F.mse_loss(z_hat, emb_train_t)
    loss_kl = F.kl_div(log_probs_hat, teacher_train_t, reduction="batchmean")
    loss=loss_kl*alpha+loss_emb*beta
    loss.backward()
    opt1.step()
    opt1.zero_grad()
    opt2.step()
    opt2.zero_grad()

    #loss = alpha * loss_emb + beta * loss_kl
    #loss.backward()

    if (epoch + 1) % 200 == 0:
        print(
            f"epoch={epoch+1:4d}  "
            f"loss={loss.item():.6f}  "
            f"emb={loss_emb.item():.6f}  "
            f"kl={loss_kl.item():.6f}"
        )


In [ ]:
with torch.no_grad():
    _, logits_test = student(X_test_t)
    student_proba_test = F.softmax(logits_test, dim=1).cpu().numpy()

teacher_pred = teacher_proba_test.argmax(axis=1)
student_pred = student_proba_test.argmax(axis=1)

teacher_acc = (teacher_pred == y_test).mean()
student_acc = (student_pred == y_test).mean()
agreement = (teacher_pred == student_pred).mean()

kl = np.mean(
    np.sum(
        teacher_proba_test
        * (
            np.log(teacher_proba_test + 1e-9)
            - np.log(student_proba_test + 1e-9)
        ),
        axis=1,
    )
)

In [ ]:

print("\nRESULTS")
print(f"Teacher acc       : {teacher_acc:.4f}")
print(f"Student acc       : {student_acc:.4f}")
print(f"Teacher agreement : {agreement:.4f}")
print(f"KL(T || S)        : {kl:.6f}")

## Evaluate every classification dataset

The single-dataset run above (breast cancer) is generalized below into a sweep over **every**
classification dataset in the benchmark suite (`benchmark_datasets.OpenMLBenchmark`). For each
dataset and split it fits the TabPFN teacher, distills the `StudentModel` (encoder regressed onto
the TabPFN embedding with MSE + decoder matched to the teacher probabilities with KL), and also
trains a **from-scratch baseline**: the *same architecture* trained directly on the hard labels with
cross-entropy (no teacher, no embedding). The baseline shows how much the distillation actually adds
over plain supervised learning at equal capacity.

Recorded per dataset:

- `teacher_acc` / `student_acc` / `baseline_acc` — test accuracy of each
- `agreement` — fraction of test rows where student and teacher predict the same class
- `teacher_auc` / `student_auc` / `baseline_auc` — ROC AUC (one-vs-rest macro for multiclass)
- `kl` — mean `KL(teacher || student)` on the test set

Everything is unified over the number of classes (softmax + KL throughout), so the same code handles
binary and multiclass datasets. Results are aggregated into a per-dataset table and an overall summary.


In [ ]:
import pandas as pd
from benchmark_datasets import OpenMLBenchmark, _roc_auc

benchmarks = OpenMLBenchmark("classification")
STUDENT_METRICS = [
    "teacher_acc", "student_acc", "baseline_acc", "agreement",
    "teacher_auc", "student_auc", "baseline_auc", "kl",
]


def _student_proba(student, x_scaler, X):
    """Class probabilities from a trained StudentModel (uses X only -- no teacher at inference)."""
    with torch.no_grad():
        _, logits = student(torch.tensor(x_scaler.transform(X), dtype=torch.float32))
        return F.softmax(logits, dim=1).cpu().numpy()


def _fit_distilled_student(X_train, y_train, n_classes, *, epochs, alpha, beta, lr, weight_decay):
    """Fit TabPFN + distill the StudentModel (encoder->embedding MSE, decoder->teacher KL).

    Returns ``(teacher, student, x_scaler, emb_dim)``.
    """
    teacher = TabPFNClassifier(n_estimators=1, random_state=42).fit(X_train, y_train)
    embedder = TabPFNEmbedding(model=teacher, n_fold=0)
    emb_train = embedder.fit_transform(X_train, y_train)
    if emb_train.ndim == 3:
        emb_train = emb_train[0]

    x_scaler = StandardScaler().fit(X_train)
    emb_scaler = StandardScaler().fit(emb_train)
    X_train_t = torch.tensor(x_scaler.transform(X_train), dtype=torch.float32)
    emb_train_t = torch.tensor(emb_scaler.transform(emb_train), dtype=torch.float32)
    teacher_train_t = torch.tensor(teacher.predict_proba(X_train), dtype=torch.float32)

    student = StudentModel(X_train.shape[1], emb_train.shape[1], n_classes)
    #opt = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=weight_decay)
    opt1 = torch.optim.AdamW(student.encoder.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.AdamW(student.decoder.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epochs):
        opt1.zero_grad()
        opt2.zero_grad()
        z_hat, logits_hat = student(X_train_t)
        log_probs_hat = F.log_softmax(logits_hat, dim=1)
        #loss = (alpha * F.mse_loss(z_hat, emb_train_t)
        #        + beta * F.kl_div(F.log_softmax(logits_hat, dim=1), teacher_train_t, reduction="batchmean"))
        loss_emb = F.mse_loss(z_hat, emb_train_t)
        loss_kl = F.kl_div(log_probs_hat, teacher_train_t, reduction="batchmean")
        loss = loss_emb*alpha+loss_kl*beta
        loss.backward()
        opt1.step()
        opt1.zero_grad()
        opt2.step()
        opt2.zero_grad()
        #loss.backward()
        #opt.step()
    return teacher, student, x_scaler, emb_train.shape[1]


def _fit_scratch_student(X_train, y_train, n_classes, emb_dim, *, epochs, lr, weight_decay):
    """Baseline: the *same architecture* as the distilled student, but trained directly on the
    hard labels with cross-entropy (no teacher, no embedding). Isolates how much the distillation
    actually adds over plain supervised learning at equal capacity.
    """
    x_scaler = StandardScaler().fit(X_train)
    X_train_t = torch.tensor(x_scaler.transform(X_train), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)

    student = StudentModel(X_train.shape[1], emb_dim, n_classes)
    opt = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=weight_decay)
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad()
        _, logits = student(X_train_t)
        loss = ce(logits, y_train_t)
        loss.backward()
        opt.step()
    return student, x_scaler


def evaluate_student_on_dataset(
    ds, *, n_splits=5, train_size=0.3, epochs=1000, alpha=1.0, beta=1.0, lr=1e-3, weight_decay=1e-4
):
    """Distill the student against TabPFN across repeated splits of one ``LoadedDataset``, and
    compare against a same-architecture from-scratch baseline. Returns ``{metric: [...]}`` with
    one entry per split.
    """
    X, y = ds.X, ds.y
    classes = np.unique(y)
    n_classes = len(classes)
    out = {k: [] for k in STUDENT_METRICS}

    for train_idx, test_idx in benchmarks.splits(
        X, y, task="classification", n_splits=n_splits, train_size=train_size
    ):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        teacher, student, x_scaler, emb_dim = _fit_distilled_student(
            X_train, y_train, n_classes,
            epochs=epochs, alpha=alpha, beta=beta, lr=lr, weight_decay=weight_decay,
        )
        baseline, base_scaler = _fit_scratch_student(
            X_train, y_train, n_classes, emb_dim, epochs=epochs, lr=lr, weight_decay=weight_decay,
        )

        teacher_proba_test = teacher.predict_proba(X_test)
        student_proba_test = _student_proba(student, x_scaler, X_test)
        baseline_proba_test = _student_proba(baseline, base_scaler, X_test)

        teacher_pred = teacher_proba_test.argmax(axis=1)
        student_pred = student_proba_test.argmax(axis=1)
        baseline_pred = baseline_proba_test.argmax(axis=1)

        out["teacher_acc"].append((teacher_pred == y_test).mean())
        out["student_acc"].append((student_pred == y_test).mean())
        out["baseline_acc"].append((baseline_pred == y_test).mean())
        out["agreement"].append((teacher_pred == student_pred).mean())
        out["teacher_auc"].append(_roc_auc(y_test, teacher_proba_test, classes))
        out["student_auc"].append(_roc_auc(y_test, student_proba_test, classes))
        out["baseline_auc"].append(_roc_auc(y_test, baseline_proba_test, classes))
        out["kl"].append(float(np.mean(np.sum(
            teacher_proba_test
            * (np.log(teacher_proba_test + 1e-9) - np.log(student_proba_test + 1e-9)),
            axis=1,
        ))))

    return out


def evaluate_all_classification_students(datasets=None, *, verbose=True, **kwargs):
    """Distill the student over every classification dataset, with a from-scratch baseline, and
    aggregate. Returns ``(per_dataset, summary)`` DataFrames. ``**kwargs`` are forwarded to
    ``evaluate_student_on_dataset`` (``n_splits``, ``train_size``, ``epochs``, ``alpha``, ``beta``,
    ``lr``, ``weight_decay``).
    """
    specs = benchmarks.specs
    if datasets is not None:
        wanted = set(datasets)
        specs = [s for s in specs if s.name in wanted]

    rows = []
    for spec in specs:
        ds = benchmarks.load(spec)
        try:
            res = evaluate_student_on_dataset(ds, **kwargs)
        except Exception as exc:                       # keep the sweep going on a bad dataset
            print(f"[skip] {spec.name}: {exc}")
            continue
        row = {"dataset": spec.name, "n_rows": spec.n_rows, "n_classes": ds.n_classes}
        for k in STUDENT_METRICS:
            row[k] = float(np.nanmean(res[k]))
        rows.append(row)
        if verbose:
            print(
                f"{spec.name:24s} (k={ds.n_classes}) "
                f"teacher={row['teacher_acc']:.3f} student={row['student_acc']:.3f} "
                f"baseline={row['baseline_acc']:.3f} agree={row['agreement']:.3f} kl={row['kl']:.3f}"
            )

    per_dataset = pd.DataFrame(rows)
    summary = pd.DataFrame(
        {
            "metric": STUDENT_METRICS,
            "mean": [per_dataset[k].mean() for k in STUDENT_METRICS],
            "std": [per_dataset[k].std() for k in STUDENT_METRICS],
        }
    )
    return per_dataset, summary


def leakage_report(datasets=None, *, train_size=0.6, epochs=400, seed=0, leak_margin=0.10):
    """Data-leakage diagnostics for the student pipeline (one split per dataset).

    Reports per dataset:
      - ``dup_rows``  : exact-duplicate test rows that also appear in train (row-level leakage)
      - ``chance``    : majority-class accuracy on the test set
      - ``student``   : distilled-student test accuracy (trained on the real labels)
      - ``permuted``  : distilled-student test accuracy when the TRAIN labels are shuffled

    The permutation test is decisive: with the X->y relationship destroyed, a leak-free pipeline
    can only score at chance on the real test labels. ``leak?`` flags ``permuted`` exceeding
    ``chance`` by more than ``leak_margin`` (i.e. the model is exploiting test information).
    """
    names = [s.name for s in benchmarks.specs] if datasets is None else list(datasets)
    rng = np.random.default_rng(seed)
    rows = []
    for name in names:
        ds = benchmarks.load(name)
        X, y = ds.X, ds.y
        n_classes = len(np.unique(y))
        train_idx, test_idx = next(
            benchmarks.splits(X, y, task="classification", n_splits=2, train_size=train_size)
        )
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        train_rows = set(map(tuple, np.round(X_train, 6)))
        dup = int(sum(tuple(r) in train_rows for r in np.round(X_test, 6)))
        chance = float(np.bincount(y_test, minlength=n_classes).max() / len(y_test))

        _, student, xs, _ = _fit_distilled_student(
            X_train, y_train, n_classes, epochs=epochs, alpha=1.0, beta=1.0, lr=1e-3, weight_decay=1e-4
        )
        student_acc = float((_student_proba(student, xs, X_test).argmax(1) == y_test).mean())

        y_perm = rng.permutation(y_train)
        _, student_p, xsp, _ = _fit_distilled_student(
            X_train, y_perm, n_classes, epochs=epochs, alpha=1.0, beta=1.0, lr=1e-3, weight_decay=1e-4
        )
        permuted_acc = float((_student_proba(student_p, xsp, X_test).argmax(1) == y_test).mean())

        leak = permuted_acc > chance + leak_margin
        rows.append({
            "dataset": name, "n_train": len(train_idx), "n_test": len(test_idx),
            "dup_rows": dup, "chance": chance, "student": student_acc,
            "permuted": permuted_acc, "leak?": leak,
        })
        print(
            f"{name:18s} dup={dup:3d} chance={chance:.3f} student={student_acc:.3f} "
            f"permuted={permuted_acc:.3f}  {'<-- LEAK' if leak else 'ok'}"
        )
    return pd.DataFrame(rows)


In [ ]:
# Run the student distillation (+ from-scratch baseline) over every classification dataset.
# Heavy: ~50 datasets x n_splits x (TabPFN fit + embedding + epochs of the 256-wide student x2).
# Pass e.g. datasets=["sonar", "wine"] or a smaller epochs/n_splits for a quick smoke test.
per_dataset, summary = evaluate_all_classification_students()

print("\n=== Aggregate across all classification datasets ===")
display(summary.round(4))
display(per_dataset.round(4))

# Mean ROC AUC across all datasets for the teacher, student, and baseline.
auc_means = summary.set_index("metric")["mean"]
print("\n=== Mean ROC AUC across all classification datasets ===")
print(f"teacher  : {auc_means['teacher_auc']:.4f}")
print(f"student  : {auc_means['student_auc']:.4f}")
print(f"baseline : {auc_means['baseline_auc']:.4f}")


## Data-leakage check

The distilled student sometimes scores *above* the teacher, which is worth scrutinising. Two checks
below rule out leakage:

1. **Duplicate rows** — count test rows that appear verbatim in the training set (a common source of
   inflated scores in small OpenML datasets).
2. **Label-permutation test** (decisive) — re-run the whole pipeline with the **training labels
   shuffled**. This destroys the `X → y` relationship, so a leak-free model can only score at
   *chance* on the real test labels. If accuracy stays high, the model is exploiting test
   information.

Structurally there is no test-label path: the teacher and the embedder are fit on the training split
only, and at inference the student maps `X_test → encoder → decoder` with its own weights (it never
runs TabPFN on the test rows). The checks confirm this empirically. The reason the student looks
strong is mundane — the teacher is deliberately weakened (`n_estimators=1`), and these small datasets
are easy for a 256-wide MLP, as the from-scratch `baseline_acc` in the sweep above shows (it matches
or beats the distilled student).


In [ ]:
# Leakage diagnostics on a representative subset (binary + multiclass, easy + hard).
# `permuted` should sit near `chance`; `leak?` should be False everywhere.
leak_df = leakage_report(["sonar", "wine", "wdbc", "diabetes", "tic-tac-toe"], epochs=400)
display(leak_df.round(3))
